In [1]:
!python verify_dataset.py

AEGIS 4.0 SYNTHETIC DATASET: VERIFIED CORE INTEGRITY
Purchase orders: 26000
Late receipt rate: 27.35%
Years: {'2024': 10695, '2025': 10562, '2026': 4743}
Monthly component inventory records: 3600
Inventory records with shortages: 628
PASS: Primary-key uniqueness, sourcing options, temporal sanity, component inventory conservation
Reminder: Integrity checks do NOT establish real-world validity or forecast performance.


In [2]:

import pandas as pd
from pathlib import Path

# Load our original AEGIS purchase-order data
file_path = Path("data") / "purchase_orders.csv"

df = pd.read_csv(file_path, parse_dates=["order_date"])

print("=" * 55)
print("AEGIS 4.0 - PURCHASE ORDER DATA PROFILING")
print("=" * 55)

print("\n1. DATASET DIMENSIONS")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("\n2. ALL AVAILABLE COLUMNS")
for i, column in enumerate(df.columns, 1):
    print(f"{i}. {column}")

print("\n3. MISSING VALUES")
print(df.isnull().sum().sort_values(ascending=False).head(15))

print("\n4. TARGET DISTRIBUTION")
print(df["late_receipt_flag"].value_counts())
print(
    "Late receipt percentage:",
    round(df["late_receipt_flag"].mean() * 100, 2)
)

print("\n5. TEMPORAL DATA SPLIT")
train = df[df["order_date"] < "2025-07-01"]
validation = df[
    (df["order_date"] >= "2025-07-01") &
    (df["order_date"] < "2026-01-01")
]
test = df[df["order_date"] >= "2026-01-01"]

for name, subset in [
    ("Training", train),
    ("Validation", validation),
    ("Testing", test)
]:
    print(
        name,
        "| Records:", len(subset),
        "| Late rate:",
        round(subset["late_receipt_flag"].mean() * 100, 2),
        "%"
    )

print("\n6. FIRST FIVE RECORDS")
display(df.head())


AEGIS 4.0 - PURCHASE ORDER DATA PROFILING

1. DATASET DIMENSIONS
Rows: 26000
Columns: 21

2. ALL AVAILABLE COLUMNS
1. purchase_order_id
2. order_date
3. supplier_id
4. component_id
5. plant_id
6. lane_id
7. order_quantity_units
8. quoted_unit_cost_usd
9. promised_lead_days
10. promised_receipt_date
11. urgent_order_flag
12. regional_alert_available_at_order
13. regional_alert_level_at_order
14. capacity_utilization_at_order
15. logistics_congestion_at_order
16. actual_receipt_date
17. actual_lead_days
18. days_late
19. late_receipt_flag
20. supplier_completed_orders_past_90d
21. supplier_late_rate_past_90d

3. MISSING VALUES
purchase_order_id                    0
order_date                           0
supplier_id                          0
component_id                         0
plant_id                             0
lane_id                              0
order_quantity_units                 0
quoted_unit_cost_usd                 0
promised_lead_days                   0
promised_receipt

,purchase_order_id,order_date,supplier_id,component_id,plant_id,lane_id,order_quantity_units,quoted_unit_cost_usd,promised_lead_days,promised_receipt_date,...,regional_alert_available_at_order,regional_alert_level_at_order,capacity_utilization_at_order,logistics_congestion_at_order,actual_receipt_date,actual_lead_days,days_late,late_receipt_flag,supplier_completed_orders_past_90d,supplier_late_rate_past_90d
0,PO-0000119,2024-01-01,SUP-038,CMP-015,PLT-2,LN-SP-SUP-038-PLT-2,738,230.99,24,2024-01-25,...,0,0,0.431,0.545,2024-01-25,24,0,0,0,0.2
1,PO-0001109,2024-01-01,SUP-046,CMP-029,PLT-2,LN-SP-SUP-046-PLT-2,314,28.63,23,2024-01-24,...,0,0,0.590,0.135,2024-01-29,28,5,1,0,0.2
2,PO-0002581,2024-01-01,SUP-011,CMP-025,PLT-2,LN-SP-SUP-011-PLT-2,613,138.61,13,2024-01-14,...,0,0,0.569,0.260,2024-01-14,13,0,0,0,0.2
3,PO-0002660,2024-01-01,SUP-030,CMP-027,PLT-2,LN-SP-SUP-030-PLT-2,237,181.89,20,2024-01-21,...,0,0,0.847,0.423,2024-01-21,20,0,0,0,0.2
4,PO-0004013,2024-01-01,SUP-015,CMP-024,PLT-2,LN-SP-SUP-015-PLT-2,211,141.28,35,2024-02-05,...,0,0,0.813,0.559,2024-02-12,42,7,1,0,0.2


In [3]:

import pandas as pd
import numpy as np

audit = df.copy()

date_columns = [
    "order_date",
    "promised_receipt_date",
    "actual_receipt_date"
]

for col in date_columns:
    audit[col] = pd.to_datetime(audit[col], errors="coerce")

# Recalculate actual lateness independently
calculated_days_late = (
    audit["actual_receipt_date"] -
    audit["promised_receipt_date"]
).dt.days

calculated_target = (calculated_days_late > 0).astype(int)

print("=" * 55)
print("AEGIS 4.0 - TARGET AND LEAKAGE AUDIT")
print("=" * 55)

print("\n1. INVALID OR MISSING DATES")
for col in date_columns:
    print(col, ":", audit[col].isna().sum())

print("\n2. DATE CHRONOLOGY")
print(
    "Promised receipt before order:",
    (audit["promised_receipt_date"] < audit["order_date"]).sum()
)
print(
    "Actual receipt before order:",
    (audit["actual_receipt_date"] < audit["order_date"]).sum()
)

print("\n3. TARGET CONSISTENCY")
print(
    "Target mismatches:",
    (calculated_target != audit["late_receipt_flag"]).sum()
)

print("\n4. DAYS-LATE CONSISTENCY")
print(
    "Differences from calculated days late:",
    (
        audit["days_late"] !=
        calculated_days_late.clip(lower=0)
    ).sum()
)

print("\n5. POTENTIALLY LEAKY VARIABLES")
post_outcome_columns = [
    "actual_receipt_date",
    "actual_lead_days",
    "days_late",
    "late_receipt_flag"
]

for col in post_outcome_columns:
    print(col, "-> Outcome-related; exclude from predictors")

print("\n6. HISTORICAL FEATURE RANGES")
historical_features = [
    "supplier_completed_orders_past_90d",
    "supplier_late_rate_past_90d"
]

print(audit[historical_features].describe())

print("\n7. DATE COVERAGE")
print("First order:", audit["order_date"].min())
print("Last order:", audit["order_date"].max())
print("Last actual receipt:", audit["actual_receipt_date"].max())

print("\nAudit complete.")


AEGIS 4.0 - TARGET AND LEAKAGE AUDIT

1. INVALID OR MISSING DATES
order_date : 0
promised_receipt_date : 0
actual_receipt_date : 0

2. DATE CHRONOLOGY
Promised receipt before order: 0
Actual receipt before order: 0

3. TARGET CONSISTENCY
Target mismatches: 0

4. DAYS-LATE CONSISTENCY
Differences from calculated days late: 0

5. POTENTIALLY LEAKY VARIABLES
actual_receipt_date -> Outcome-related; exclude from predictors
actual_lead_days -> Outcome-related; exclude from predictors
days_late -> Outcome-related; exclude from predictors
late_receipt_flag -> Outcome-related; exclude from predictors

6. HISTORICAL FEATURE RANGES
       supplier_completed_orders_past_90d  supplier_late_rate_past_90d
count                        26000.000000                 26000.000000
mean                            50.834423                     0.251181
std                             24.434277                     0.084230
min                              0.000000                     0.062500
25%             

In [4]:

import pandas as pd

# Work with a copy, never modify the original dataset
clean = df.copy()

clean["order_date"] = pd.to_datetime(clean["order_date"])
clean["actual_receipt_date"] = pd.to_datetime(
    clean["actual_receipt_date"]
)

train_cutoff = pd.Timestamp("2025-07-01")
test_cutoff = pd.Timestamp("2026-01-01")

# Training: Only outcomes available before validation starts
train_safe = clean[
    (clean["order_date"] < train_cutoff) &
    (clean["actual_receipt_date"] < train_cutoff)
].copy()

# Validation: Only outcomes available before testing starts
validation_safe = clean[
    (clean["order_date"] >= train_cutoff) &
    (clean["order_date"] < test_cutoff) &
    (clean["actual_receipt_date"] < test_cutoff)
].copy()

# Testing: Orders placed from January 2026 onward
test_safe = clean[
    clean["order_date"] >= test_cutoff
].copy()

print("=" * 55)
print("AEGIS 4.0 - LEAKAGE-SAFE DATA SPLITTING")
print("=" * 55)

for name, data in [
    ("TRAINING", train_safe),
    ("VALIDATION", validation_safe),
    ("TESTING", test_safe)
]:
    print(f"\n{name}")
    print("Records:", len(data))
    print(
        "Late receipt rate:",
        round(data["late_receipt_flag"].mean() * 100, 2),
        "%"
    )

# Validate historical outcome availability
assert (
    train_safe["actual_receipt_date"] < train_cutoff
).all()

assert (
    validation_safe["actual_receipt_date"] < test_cutoff
).all()

# Check for overlapping orders
train_ids = set(train_safe["purchase_order_id"])
val_ids = set(validation_safe["purchase_order_id"])
test_ids = set(test_safe["purchase_order_id"])

assert train_ids.isdisjoint(val_ids)
assert train_ids.isdisjoint(test_ids)
assert val_ids.isdisjoint(test_ids)

print("\nAll outcome-availability checks passed.")
print("No overlapping purchase orders across splits.")
print("Original dataset remains unchanged.")


AEGIS 4.0 - LEAKAGE-SAFE DATA SPLITTING

TRAINING
Records: 15253
Late receipt rate: 27.04 %

VALIDATION
Records: 4598
Late receipt rate: 20.49 %

TESTING
Records: 4743
Late receipt rate: 34.2 %

All outcome-availability checks passed.
No overlapping purchase orders across splits.
Original dataset remains unchanged.


In [5]:

import pandas as pd

print("=" * 55)
print("AEGIS 4.0 - SPLIT SELECTION BIAS AUDIT")
print("=" * 55)

audit_df = df.copy()

audit_df["order_date"] = pd.to_datetime(
    audit_df["order_date"]
)
audit_df["actual_receipt_date"] = pd.to_datetime(
    audit_df["actual_receipt_date"]
)

periods = [
    ("TRAINING", "2024-01-01", "2025-07-01"),
    ("VALIDATION", "2025-07-01", "2026-01-01")
]

for name, start, end in periods:

    cutoff = pd.Timestamp(end)

    period_data = audit_df[
        (audit_df["order_date"] >= pd.Timestamp(start)) &
        (audit_df["order_date"] < cutoff)
    ].copy()

    retained = period_data[
        period_data["actual_receipt_date"] < cutoff
    ]

    excluded = period_data[
        period_data["actual_receipt_date"] >= cutoff
    ]

    print(f"\n{name} PERIOD")
    print("-" * 40)

    print("Original records:", len(period_data))
    print("Retained records:", len(retained))
    print("Excluded records:", len(excluded))

    print(
        "Exclusion percentage:",
        round(len(excluded) / len(period_data) * 100, 2),
        "%"
    )

    for label, data in [
        ("Original", period_data),
        ("Retained", retained),
        ("Excluded", excluded)
    ]:
        print(
            f"{label} late rate:",
            round(data["late_receipt_flag"].mean() * 100, 2)
            if len(data) else "N/A"
        )

    print(
        "Excluded average actual lead time:",
        round(excluded["actual_lead_days"].mean(), 2)
        if len(excluded) else "N/A"
    )

print("\nAUDIT COMPLETE")


AEGIS 4.0 - SPLIT SELECTION BIAS AUDIT

TRAINING PERIOD
----------------------------------------
Original records: 15969
Retained records: 15253
Excluded records: 716
Exclusion percentage: 4.48 %
Original late rate: 27.17
Retained late rate: 27.04
Excluded late rate: 29.89
Excluded average actual lead time: 25.94

VALIDATION PERIOD
----------------------------------------
Original records: 5288
Retained records: 4598
Excluded records: 690
Exclusion percentage: 13.05 %
Original late rate: 21.75
Retained late rate: 20.49
Excluded late rate: 30.14
Excluded average actual lead time: 26.37

AUDIT COMPLETE


In [6]:

import pandas as pd

print("=" * 60)
print("AEGIS 4.0 - FIXED EMBARGO TEMPORAL SPLIT")
print("=" * 60)

data = df.copy()

data["order_date"] = pd.to_datetime(data["order_date"])
data["actual_receipt_date"] = pd.to_datetime(
    data["actual_receipt_date"]
)

validation_start = pd.Timestamp("2025-07-01")
test_start = pd.Timestamp("2026-01-01")

embargo = pd.Timedelta(days=75)

train_end = validation_start - embargo
validation_end = test_start - embargo

# Select records using ORDER DATE ONLY
train_final = data[
    data["order_date"] < train_end
].copy()

validation_final = data[
    (data["order_date"] >= validation_start) &
    (data["order_date"] < validation_end)
].copy()

test_final = data[
    data["order_date"] >= test_start
].copy()

print("\n1. DATASET PARTITIONS")

for name, subset in [
    ("TRAINING", train_final),
    ("VALIDATION", validation_final),
    ("TESTING", test_final)
]:
    print(f"\n{name}")
    print("Records:", len(subset))
    print(
        "Late rate:",
        round(subset["late_receipt_flag"].mean() * 100, 2),
        "%"
    )
    print("Earliest order:", subset["order_date"].min())
    print("Latest order:", subset["order_date"].max())

print("\n2. OUTCOME AVAILABILITY CHECK")

train_unavailable = (
    train_final["actual_receipt_date"] >= validation_start
).sum()

val_unavailable = (
    validation_final["actual_receipt_date"] >= test_start
).sum()

print("Unavailable training outcomes:", train_unavailable)
print("Unavailable validation outcomes:", val_unavailable)

print("\n3. DATASET COVERAGE")

total_used = (
    len(train_final) +
    len(validation_final) +
    len(test_final)
)

print("Original records:", len(data))
print("Retained records:", total_used)
print("Gap-period records:", len(data) - total_used)

print("\n4. UNIQUENESS CHECK")

ids = pd.concat([
    train_final["purchase_order_id"],
    validation_final["purchase_order_id"],
    test_final["purchase_order_id"]
])

print("Duplicate IDs across splits:", ids.duplicated().sum())

print("\n5. FINAL STATUS")

if (
    train_unavailable == 0
    and val_unavailable == 0
    and ids.duplicated().sum() == 0
):
    print("PASS: Proposed chronological split is valid.")
else:
    print("REVIEW REQUIRED: Split needs adjustment.")

print("\nAudit complete.")


AEGIS 4.0 - FIXED EMBARGO TEMPORAL SPLIT

1. DATASET PARTITIONS

TRAINING
Records: 13713
Late rate: 26.56 %
Earliest order: 2024-01-01 00:00:00
Latest order: 2025-04-16 00:00:00

VALIDATION
Records: 3122
Late rate: 22.01 %
Earliest order: 2025-07-01 00:00:00
Latest order: 2025-10-17 00:00:00

TESTING
Records: 4743
Late rate: 34.2 %
Earliest order: 2026-01-01 00:00:00
Latest order: 2026-06-13 00:00:00

2. OUTCOME AVAILABILITY CHECK
Unavailable training outcomes: 0
Unavailable validation outcomes: 0

3. DATASET COVERAGE
Original records: 26000
Retained records: 21578
Gap-period records: 4422

4. UNIQUENESS CHECK
Duplicate IDs across splits: 0

5. FINAL STATUS
PASS: Proposed chronological split is valid.

Audit complete.


In [7]:

import pandas as pd
import numpy as np

print("=" * 60)
print("AEGIS 4.0 - FEATURE AVAILABILITY AUDIT")
print("=" * 60)

# Candidate features available or potentially available
# at the moment a purchase order is placed.

candidate_features = [
    "supplier_id",
    "component_id",
    "plant_id",
    "lane_id",
    "order_quantity_units",
    "quoted_unit_cost_usd",
    "promised_lead_days",
    "urgent_order_flag",
    "regional_alert_available_at_order",
    "regional_alert_level_at_order",
    "capacity_utilization_at_order",
    "logistics_congestion_at_order",
    "supplier_completed_orders_past_90d",
    "supplier_late_rate_past_90d"
]

# Outcome-related columns that must not enter X
prohibited_features = [
    "actual_receipt_date",
    "actual_lead_days",
    "days_late",
    "late_receipt_flag"
]

print("\n1. FEATURE EXISTENCE")

missing_columns = [
    col for col in candidate_features
    if col not in df.columns
]

print("Candidate features:", len(candidate_features))
print("Missing feature columns:", missing_columns)

if missing_columns:
    raise ValueError("Required feature columns are missing.")

print("\n2. LEAKAGE EXCLUSION CHECK")

leakage_found = set(candidate_features).intersection(
    prohibited_features
)

print("Prohibited features selected:", leakage_found)

assert not leakage_found, "Outcome leakage detected!"

print("\n3. MISSING AND INFINITE VALUES")

for name, subset in [
    ("Training", train_final),
    ("Validation", validation_final),
    ("Testing", test_final)
]:
    X = subset[candidate_features]

    missing_count = int(X.isna().sum().sum())

    numeric = X.select_dtypes(include=np.number)
    infinite_count = int(
        np.isinf(numeric.to_numpy()).sum()
    )

    print(
        name,
        "| Missing:", missing_count,
        "| Infinite:", infinite_count
    )

print("\n4. BASIC FEATURE VALIDITY")

checks = {
    "Nonpositive order quantities": (
        df["order_quantity_units"] <= 0
    ).sum(),

    "Nonpositive unit costs": (
        df["quoted_unit_cost_usd"] <= 0
    ).sum(),

    "Nonpositive promised lead times": (
        df["promised_lead_days"] <= 0
    ).sum(),

    "Negative historical order counts": (
        df["supplier_completed_orders_past_90d"] < 0
    ).sum(),

    "Historical late rates outside 0 to 1": (
        ~df["supplier_late_rate_past_90d"]
        .between(0, 1)
    ).sum()
}

for check, count in checks.items():
    print(check, ":", int(count))

print("\n5. FEATURE DATA TYPES")

print(df[candidate_features].dtypes.to_string())

print("\n6. NUMERIC FEATURE RANGES")

numeric_features = df[candidate_features].select_dtypes(
    include=np.number
)

print(
    numeric_features
    .agg(["min", "max"])
    .transpose()
    .to_string()
)

print("\n7. HISTORICAL OUTCOME BOUNDARIES")

print(
    "Latest training receipt:",
    train_final["actual_receipt_date"].max()
)

print(
    "Latest validation receipt:",
    validation_final["actual_receipt_date"].max()
)

print("\nFEATURE AUDIT COMPLETE")


AEGIS 4.0 - FEATURE AVAILABILITY AUDIT

1. FEATURE EXISTENCE
Candidate features: 14
Missing feature columns: []

2. LEAKAGE EXCLUSION CHECK
Prohibited features selected: set()

3. MISSING AND INFINITE VALUES
Training | Missing: 0 | Infinite: 0
Validation | Missing: 0 | Infinite: 0
Testing | Missing: 0 | Infinite: 0

4. BASIC FEATURE VALIDITY
Nonpositive order quantities : 0
Nonpositive unit costs : 0
Nonpositive promised lead times : 0
Negative historical order counts : 0
Historical late rates outside 0 to 1 : 0

5. FEATURE DATA TYPES
supplier_id                               str
component_id                              str
plant_id                                  str
lane_id                                   str
order_quantity_units                    int64
quoted_unit_cost_usd                  float64
promised_lead_days                      int64
urgent_order_flag                       int64
regional_alert_available_at_order       int64
regional_alert_level_at_order           int64

In [8]:

import pandas as pd
import numpy as np

print("=" * 60)
print("AEGIS 4.0 - DATA DISTRIBUTION AND DRIFT AUDIT")
print("=" * 60)

# Use our previously verified temporal partitions
datasets = {
    "Training": train_final,
    "Validation": validation_final,
    "Testing": test_final
}

# Operational indicators to investigate
features = [
    "capacity_utilization_at_order",
    "logistics_congestion_at_order",
    "regional_alert_available_at_order",
    "regional_alert_level_at_order",
    "urgent_order_flag",
    "promised_lead_days",
    "supplier_late_rate_past_90d"
]

print("\n1. FEATURE MEANS ACROSS PERIODS")

summary = pd.DataFrame({
    name: subset[features].mean()
    for name, subset in datasets.items()
})

print(summary.round(4).to_string())

print("\n2. STANDARDIZED FEATURE SHIFTS")

# Standardize mean changes using training variability
train_std = train_final[features].std(ddof=1)

safe_std = train_std.replace(0, np.nan)

mean_shift = (
    summary["Testing"] - summary["Training"]
) / safe_std

shift_table = pd.DataFrame({
    "Train Mean": summary["Training"],
    "Test Mean": summary["Testing"],
    "Std Mean Shift": mean_shift
})

print(shift_table.round(4).to_string())

print("\n3. TARGET RATE COMPARISON")

for name, subset in datasets.items():
    late_rate = subset["late_receipt_flag"].mean()

    print(
        f"{name}: {late_rate * 100:.2f}% "
        f"({len(subset):,} records)"
    )

print("\n4. SUPPLIER COVERAGE")

train_suppliers = set(train_final["supplier_id"])

for name, subset in datasets.items():

    suppliers = set(subset["supplier_id"])

    unseen = suppliers - train_suppliers

    print(
        name,
        "| Suppliers:", len(suppliers),
        "| Unseen suppliers:", len(unseen)
    )

print("\n5. ALERT FREQUENCIES")

for name, subset in datasets.items():

    alert_percentage = (
        subset["regional_alert_available_at_order"]
        .mean() * 100
    )

    print(
        name,
        "| Regional alert frequency:",
        round(alert_percentage, 2),
        "%"
    )

print("\nDISTRIBUTION AUDIT COMPLETE")


AEGIS 4.0 - DATA DISTRIBUTION AND DRIFT AUDIT

1. FEATURE MEANS ACROSS PERIODS
                                   Training  Validation  Testing
capacity_utilization_at_order        0.7264      0.5705   0.9060
logistics_congestion_at_order        0.3674      0.2346   0.4648
regional_alert_available_at_order    0.3662      0.4289   0.4099
regional_alert_level_at_order        0.4631      0.5330   0.5385
urgent_order_flag                    0.1940      0.1887   0.1933
promised_lead_days                  23.1972     23.0881  23.1697
supplier_late_rate_past_90d          0.2441      0.2626   0.2708

2. STANDARDIZED FEATURE SHIFTS
                                   Train Mean  Test Mean  Std Mean Shift
capacity_utilization_at_order          0.7264     0.9060          0.7643
logistics_congestion_at_order          0.3674     0.4648          0.4825
regional_alert_available_at_order      0.3662     0.4099          0.0906
regional_alert_level_at_order          0.4631     0.5385          0.1111
urge

In [9]:

import pandas as pd
from pathlib import Path

print("=" * 60)
print("AEGIS 4.0 - CROSS-DATASET RELATIONSHIP AUDIT")
print("=" * 60)

data_dir = Path("data")

orders = pd.read_csv(
    data_dir / "purchase_orders.csv",
    parse_dates=["order_date"]
)

inventory = pd.read_csv(
    data_dir / "component_inventory_monthly.csv"
)

suppliers = pd.read_csv(
    data_dir / "suppliers.csv"
)

options = pd.read_csv(
    data_dir / "supplier_component_options.csv"
)

print("\n1. SUPPLIER NETWORK COVERAGE")

master_suppliers = set(suppliers["supplier_id"])
active_suppliers = set(options["supplier_id"])

print("Master suppliers:", len(master_suppliers))
print("Active suppliers:", len(active_suppliers))
print("Unused suppliers:", sorted(
    master_suppliers - active_suppliers
))

print("\n2. MONTHLY PROCUREMENT AND REQUIREMENTS")

orders["month"] = (
    orders["order_date"].dt.strftime("%Y-%m")
)

monthly_orders = (
    orders.groupby(
        ["month", "plant_id", "component_id"],
        as_index=False
    )["order_quantity_units"]
    .sum()
    .rename(columns={
        "order_quantity_units": "ordered_units"
    })
)

combined = inventory.merge(
    monthly_orders,
    on=["month", "plant_id", "component_id"],
    how="left",
    validate="one_to_one"
)

combined["ordered_units"] = (
    combined["ordered_units"].fillna(0)
)

correlation = combined[
    ["ordered_units", "planned_requirements_units"]
].corr().iloc[0, 1]

print("Matched inventory records:", len(combined))
print("Demand-order correlation:", round(correlation, 4))

print("\n3. INVENTORY AND SHORTAGE CONSISTENCY")

print(
    "Total component requirements:",
    int(inventory["planned_requirements_units"].sum())
)

print(
    "Total component receipts:",
    int(inventory["receipts_units"].sum())
)

print(
    "Total unmet component requirements:",
    int(inventory["unmet_component_requirements_units"].sum())
)

print(
    "Records with shortages:",
    int((
        inventory["unmet_component_requirements_units"] > 0
    ).sum())
)

print("\n4. MONTHLY ORDER COVERAGE")

print(
    "Records with no orders placed:",
    int((combined["ordered_units"] == 0).sum())
)

print(
    "Records with orders placed:",
    int((combined["ordered_units"] > 0).sum())
)

print("\nCROSS-DATASET AUDIT COMPLETE")


AEGIS 4.0 - CROSS-DATASET RELATIONSHIP AUDIT

1. SUPPLIER NETWORK COVERAGE
Master suppliers: 60
Active suppliers: 58
Unused suppliers: ['SUP-029', 'SUP-033']

2. MONTHLY PROCUREMENT AND REQUIREMENTS
Matched inventory records: 3600
Demand-order correlation: 0.0098

3. INVENTORY AND SHORTAGE CONSISTENCY
Total component requirements: 7710553
Total component receipts: 10788605
Total unmet component requirements: 1269976
Records with shortages: 628

4. MONTHLY ORDER COVERAGE
Records with no orders placed: 4
Records with orders placed: 3596

CROSS-DATASET AUDIT COMPLETE
